<a href="https://colab.research.google.com/github/Aris-1712/LLM-fine-tuning/blob/main/Non_Instructional_fine_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Process:

# -Get a base model
# -Do non instructional fine-tuning (For Domain specific knowledge) with labels
# -Do instructional fine tuning (For Q and A type answering)
# -Do RLHF for human feedback and retraining (DPO)


# Steps for dataset
# 1. get pdf, extract information from it and store it
# 2. Chunk it based on some logic, could be semantic could be html tags etc.
# 3. use huggingface library to create datasset in the correct format

In [ ]:
# install dependencies

!pip install transformers
!pip install datasets
!pip install accelerate
!pip install peft
!pip install trl
!pip install bitsandbytes
!pip install pyMuPdf

In [ ]:
# Read data from PDF

import pymupdf
data = ""
doc = pymupdf.open("/content/Financial-Report.pdf") # open a document
for page in doc: # iterate the document pages
    text = page.get_text()
    data += text

In [ ]:
# Prepare data in Datasets format for the model
dataList = []
dataList = data.split("\n \n")
for index,x in enumerate(dataList):
  x = {"text": x}
  dataList[index] = x
print(dataList)

In [ ]:
# coonvert from list Dataset format
from datasets import Dataset
dataSet = Dataset.from_list(dataList)
dataSet

In [ ]:
# Sample data
dataSet[0]

In [ ]:
# Steps further

# - Select Model
# - EOS TOKEN - End of sentence token
# - Tokenizer
# - pad token to make sure tensors or vectors have the same size.
# - tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")

# sequence basically mean list of inpute, padding true means whenther we should do padding to the inputs to make all the vectors of same size, truncation tells us whether we should truncate the inputs to max token length provided by the model

In [ ]:
# load tokenzier to convert strint to tokens

from transformers import AutoTokenizer
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

# tokenizer should be of the model itself
tokenizer = AutoTokenizer.from_pretrained(model)


In [ ]:
# set pad token and eos (end of sentence) token
if tokenizer.pad_token == None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# function to tokenize strings and add labels
#  why we need labels: Labels are used to identify the next token while the model is being trained so label has to be there
def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    for label in tokens["labels"]:
        if label == tokenizer.pad_token_id:
            label = -100
    return tokens

In [ ]:
# tokenize the dataset in batch
tokenized = dataSet.map(tokenize_fn, batched=True)

In [ ]:
# load the pretrained model in quantized manner
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model = AutoModelForCausalLM.from_pretrained(
    model,
    quantization_config=quantization_config,
    device_map="auto"
)

In [ ]:
# lets create the peft model - finetuning model based on weights
# for that we need lora config first

from peft import LoraConfig, TaskType
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)



In [ ]:
# crete the peft model using the lora config

from peft import get_peft_model
peft_model = get_peft_model(model, lora_config)


In [ ]:
# create training configs

from transformers import TrainingArguments
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
# start the training

from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

In [ ]:
path = "/content/tinyllama-lora/checkpoint-69"

In [ ]:
model = AutoModelForCausalLM.from_pretrained(adapter_path, device_map="auto")

In [ ]:
# from transformers import AutoTokenizer
# tokenizer = AutoTokenizer.from_pretrained(adapter_path)

In [ ]:
prompt = "We record deferred revenues when cash payments "
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200, # Increased max_new_tokens
    temperature=0.7,    # Decreased temperature for less randomness
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(outputs)

decoded_output = tokenizer.decode(outputs[0],skip_special_tokens=True) # Removed skip_special_tokens=True
print(f"\nDecoded Output (including special tokens):\n{decoded_output}")